# Introduction to Data Science 2026

# Week 4

In this week's exercise, we look at prompting and zero- and few-shot task settings. Below is a text generation example from https://github.com/TurkuNLP/intro-to-nlp/blob/master/text_generation_pipeline_example.ipynb demonstrating how to load a text generation pipeline with a pre-trained model and generate text with a given prompt. Your task is to load a similar pre-trained generative model and assess whether the model succeeds at a set of tasks in zero-shot, one-shot, and two-shot settings.

**Note: Downloading and running the pre-trained model locally may take some time. Alternatively, you can open and run this notebook on [Google Colab](https://colab.research.google.com/), as assumed in the following example.**

## Text generation example

This is a brief example of how to run text generation with a causal language model and `pipeline`.

Install [transformers](https://huggingface.co/docs/transformers/index) python package. This will be used to load the model and tokenizer and to run generation.

In [1]:
!pip install --quiet transformers

Import the `AutoTokenizer`, `AutoModelForCausalLM`, and `pipeline` classes. The first two support loading tokenizers and generative models from the [Hugging Face repository](https://huggingface.co/models), and the last wraps a tokenizer and a model for convenience.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import pandas as pd
import torch

/Users/nivedharengaraj/.pyenv/versions/3.11.5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load a generative model and its tokenizer. You can substitute any other generative model name here (e.g. [other TurkuNLP GPT-3 models](https://huggingface.co/models?sort=downloads&search=turkunlp%2Fgpt3) or `gpt2-large`), but note that Colab may have issues running larger models.

In [3]:
MODEL_NAME = 'gpt2-large'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

Instantiate a text generation pipeline using the tokenizer and model.

In [4]:
pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    device=model.device
)

We can now call the pipeline with a text prompt; it will take care of tokenizing, encoding, generation, and decoding:

In [5]:
output = pipe('Hello, how are you today?', max_new_tokens=25, pad_token_id=tokenizer.eos_token_id)

print(output)

[{'generated_text': "Hello, how are you today?\n\nI'm fine, thank you. I was going to say, I don't understand English very well, but I"}]


Just print the text

In [6]:
print(output[0]['generated_text'])

Hello, how are you today?

Oh, how am I?

I'm good, but it's been a long day.

[


We can also call the pipeline with any arguments that the model `generate` function supports. For details on text generation using `transformers`, see e.g. [this tutorial](https://huggingface.co/blog/how-to-generate).

Example with sampling and a higher `temperature` parameter to generate more diverse output:

In [8]:
output = pipe(
    'Hello, hows the weather?',
    do_sample=True,
    temperature=1.3,
    max_new_tokens=35,
    pad_token_id=tokenizer.eos_token_id
)

print(output[0]['generated_text'])

Hello, hows the weather? - (This means that you are getting sick of their jokes. Your partner says, "Hey, hows the weather? - You're tired of their jokes. Your partner


## Exercise 1

Your task is to assess whether a generative model succeeds in the following tasks in zero-shot, one-shot, and two-shot settings:

- binary sentiment classification (positive / negative)

- person name recognition

- two-digit addition (e.g. 11 + 22 = 33)

For example, for assessing whether a generative model can name capital cities, we could use the following prompts:

- zero-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- one-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""
- two-shot:
	>"""\
	>Identify the capital cities of countries.
	>
	>Question: What is the capital of Sweden?\
	>Answer: Stockholm
	>
	>Question: What is the capital of Denmark?\
	>Answer: Copenhagen
	>
	>Question: What is the capital of Finland?\
	>Answer:\
	>"""

You can do the tasks either in English or Finnish and use a generative model of your choice from the Hugging Face models repository, for example the following models:

- English: `gpt2-large`
- Finnish: `TurkuNLP/gpt3-finnish-large`

You can either come up with your own instructions for the tasks or use the following:

- English:
	- binary sentiment classification: "Do the following texts express a positive or negative sentiment?"
	- person name recognition: "List the person names occurring in the following texts."
	- two-digit addition: "This is a first grade math exam."
- Finnish:
	- binary sentiment classification: "Ilmaisevatko seuraavat tekstit positiivista vai negatiivista tunnetta?"
	- person name recognition: "Listaa seuraavissa teksteissä mainitut henkilönnimet."
	- two-digit addition: "Tämä on ensimmäisen luokan matematiikan koe."

Come up with at least two test cases for each of the three tasks, and come up with your own one- and two-shot examples.

In [12]:
# Define the tasks, instructions, few-shot demonstration examples, and test cases
tasks = [
    {
    'task_name': 'Binary Sentiment Classification',
    'instruction': 'Do the following dress reviews express a positive or negative sentiment?',
    
    'shots': [
        {
            'text': 'The dress is beautiful, fits perfectly, and the fabric feels soft and comfortable.',
            'label': 'Positive'
        },
        {
            'text': 'The dress looks cheap, the stitching is poor, and the fabric is uncomfortable.',
            'label': 'Negative'
        }
    ],
    
    'test_cases': [
        {
            'id': 'TC1 (Positive)',
            'text': 'I absolutely love this dress! The color is gorgeous and it fits me perfectly.',
            'expected': 'Positive'
        },
        {
            'id': 'TC2 (Negative)',
            'text': 'I was disappointed with this dress. The material feels cheap and the size is completely wrong.',
            'expected': 'Negative'
        }
    ],
    
    'format_shot': lambda s: f"Review: {s['text']}\nSentiment: {s['label']} ",
    'format_query': lambda tc: f"Review: {tc['text']}\nSentiment:",
    'max_new_tokens': 10
    },   
    {
        'task_name': 'Person Name Recognition',
        'instruction': 'List the person names occurring in the following texts.',
        'shots': [
            {'text': 'Yesterday, Marie Curie and Albert Einstein discussed physics at the international conference in Brussels.', 'label': 'Marie Curie, Albert Einstein'},
            {'text': 'During the championship tournament, Serena Williams and Roger Federer delivered remarkable performances.', 'label': 'Serena Williams, Roger Federer'}
        ],
        'test_cases': [
            {'id': 'TC1 (2 Names)', 'text': 'In the afternoon, Nivedha Rengaraj and Gagandeep Vijay met in London to discuss computational theory.', 'expected': 'Nivedha Rengaraj, Gagandeep Vijay'},
            {'id': 'TC2 (3 Names)', 'text': 'The acclaimed director Vamsi went to meeet actors Uma and Nandhu.', 'expected': 'Vamsi, Uma, Nandhu'}
        ],
        'format_shot': lambda s: f"Text: {s['text']}\nNames: {s['label']}",
        'format_query': lambda tc: f"Text: {tc['text']}\nNames:",
        'max_new_tokens': 15
    },
    {
        'task_name': 'Two-Digit Addition',
        'instruction': 'This is a first grade math exam.',
        'shots': [
            {'text': 'Question: 15 + 23 =', 'label': '38'},
            {'text': 'Question: 41 + 17 =', 'label': '58'}
        ],
        'test_cases': [
            {'id': 'TC1 (11 + 22)', 'text': 'Question: 11 + 22 =', 'expected': '33'},
            {'id': 'TC2 (35 + 42)', 'text': 'Question: 35 + 42 =', 'expected': '77'}
        ],
        'format_shot': lambda s: f"{s['text']}\nAnswer: {s['label']}",
        'format_query': lambda tc: f"{tc['text']}\nAnswer:",
        'max_new_tokens': 10
    }
]

def build_prompt(instruction, shots, test_query, k_shot=0, format_shot_fn=None):
    """Constructs 0-shot, 1-shot, or 2-shot formatted prompt string."""
    prompt_parts = [instruction, '']
    for i in range(k_shot):
        prompt_parts.append(format_shot_fn(shots[i]))
        prompt_parts.append('')
    prompt_parts.append(test_query)
    return '\n'.join(prompt_parts)

results = []

# Run evaluation across all tasks and k-shot settings
for task in tasks:
    print('=' * 75)
    print(f"TASK: {task['task_name'].upper()}")
    print('=' * 75)
    
    for tc in task['test_cases']:
        query_str = task['format_query'](tc)
        print(f"\n--- Test Case: {tc['id']} | Expected Target: '{tc['expected']}' ---")
        
        for k in [0, 1, 2]:
            prompt = build_prompt(task['instruction'], task['shots'], query_str, k_shot=k, format_shot_fn=task['format_shot'])
            res = pipe(prompt, max_new_tokens=task['max_new_tokens'], do_sample=False, pad_token_id=tokenizer.eos_token_id)
            generated_full = res[0]['generated_text']
            # Extract continuation after prompt and take first non-empty line
            completion = generated_full[len(prompt):].strip().split('\n')[0].strip()
            
            # Determine evaluation result
            expected = tc['expected']
            if completion.strip().lower() == expected.strip().lower():
                status = 'Success'
            elif any(part.strip() in completion for part in expected.split(',')):
                status = 'Partial'
            else:
                status = 'Fail'
                
            print(f"  [{k}-Shot Output]: '{completion}'  -->  Status: {status}")
            
            results.append({
                'Task': task['task_name'],
                'Test Case': tc['id'],
                'Setting': f'{k}-shot',
                'Expected Target': tc['expected'],
                'Model Output': completion,
                'Result': status
            })

# Create structured summary table
results_df = pd.DataFrame(results)
print('\n' + '=' * 75)
print('SUMMARY EVALUATION TABLE')
print('=' * 75)

summary_table = results_df.pivot_table(
    index=['Task', 'Test Case'],
    columns='Setting',
    values='Result',
    aggfunc='first'
).reset_index()

summary_table = summary_table[['Task', 'Test Case', '0-shot', '1-shot', '2-shot']]

print(summary_table.to_string(index=False))

TASK: BINARY SENTIMENT CLASSIFICATION

--- Test Case: TC1 (Positive) | Expected Target: 'Positive' ---
  [0-Shot Output]: 'I absolutely love this dress! The color is gorgeous'  -->  Status: Fail
  [1-Shot Output]: 'Positive'  -->  Status: Success
  [2-Shot Output]: 'Positive'  -->  Status: Success

--- Test Case: TC2 (Negative) | Expected Target: 'Negative' ---
  [0-Shot Output]: 'I was disappointed with this dress. The material feels'  -->  Status: Fail
  [1-Shot Output]: 'Negative'  -->  Status: Success
  [2-Shot Output]: 'Negative'  -->  Status: Success
TASK: PERSON NAME RECOGNITION

--- Test Case: TC1 (2 Names) | Expected Target: 'Nivedha Rengaraj, Gagandeep Vijay' ---
  [0-Shot Output]: 'Nivedha Rengaraj, Gagandeep Vijay,'  -->  Status: Partial
  [1-Shot Output]: 'Nivedha Rengaraj, Gagandeep Vijay'  -->  Status: Success
  [2-Shot Output]: 'Nivedha Rengaraj, Gagandeep Vijay'  -->  Status: Success

--- Test Case: TC2 (3 Names) | Expected Target: 'Vamsi, Uma, Nandhu' ---
  [0-Shot Ou

In [ ]:
###OBservation
#1. 0 shot almostly failed at all the tasks for this pretrained model than 1 and 2 shot settings mostly because it might be easier for the model to follow a pattern and perform.
#2. Task3 is failed by all prompt settings.
#3. However, this is not always the case and sometimes models under perform well under 1-2 shot settings when they are confused with a lot of examples too.

**Submit this exercise by submitting your code and your answers to the above questions as comments on the MOOC platform. You can return this Jupyter notebook (.ipynb) or .py, .R, etc depending on your programming preferences.**